In [1]:
from pathlib import Path
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder, find_peaks
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# Paths
cutout_dir = Path("../data/raw/v854_cen_dasch/cutouts")
output_dir = Path("../outputs/tables")
os.makedirs(output_dir, exist_ok=True)

# Detection parameters
FWHM = 3.0
THRESHOLD_SIGMA = 5.0
MIN_SOURCES_DAO = 5   # fall back to find_peaks if DAOStarFinder finds fewer than this
BOX_SIZE = 11         # for find_peaks

cutouts = sorted(cutout_dir.glob("*.fits"))
print(f"Found {len(cutouts)} cutouts to process")

Found 4540 cutouts to process


In [2]:
def detect_sources(fits_path):
    data = fits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))

    mean, median, std = sigma_clipped_stats(data, sigma=3.0)
    data_sub = data - median

    try:
        daofind = DAOStarFinder(
            fwhm=FWHM,
            threshold=THRESHOLD_SIGMA * std,
            sharpness_range=(0.2, 2.0)
        )
        dao_sources = daofind(data_sub)
        n_dao = 0 if dao_sources is None else len(dao_sources)
    except Exception:
        dao_sources = None
        n_dao = 0

    if n_dao >= MIN_SOURCES_DAO:
        sources = dao_sources
        algorithm = "DAOStarFinder"
        x_col, y_col = "x_centroid", "y_centroid"
    else:
        sources = find_peaks(data_sub, threshold=THRESHOLD_SIGMA * std, box_size=BOX_SIZE)
        algorithm = "find_peaks"
        x_col, y_col = "x_peak", "y_peak"

        if sources is not None and len(sources) > 0:
            from photutils.morphology import data_properties
            sharpness_vals, elongation_vals = [], []
            for row in sources:
                x, y = int(row["x_peak"]), int(row["y_peak"])
                cutout = data_sub[max(0,y-7):y+7, max(0,x-7):x+7]
                if cutout.size > 0:
                    try:
                        props = data_properties(cutout)
                        sharpness_vals.append(float(props.max_value / props.segment_fluxes[0]) if props.segment_fluxes[0] > 0 else np.nan)
                        elongation_vals.append(float(props.elongation))
                    except:
                        sharpness_vals.append(np.nan)
                        elongation_vals.append(np.nan)
                else:
                    sharpness_vals.append(np.nan)
                    elongation_vals.append(np.nan)
            sources["sharpness"] = sharpness_vals
            sources["elongation"] = elongation_vals

    return sources, algorithm, data, data_sub, std, x_col, y_col

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

from photutils.aperture import CircularAperture, aperture_photometry
from astropy.io import fits as astrofits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astroquery.simbad import Simbad

import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
import warnings

APERTURE_RADIUS = 5.0

# ----------------------------
# SIMBAD CACHE
# ----------------------------
catalog_cache = {}
SIMBAD_RADIUS = 0.25 * u.deg


def get_plate_catalog(wcs, shape):

    cx, cy = shape[1] / 2, shape[0] / 2
    ra_c, dec_c = wcs.pixel_to_world_values(cx, cy)

    ra_c = float(np.array(ra_c))
    dec_c = float(np.array(dec_c))

    key = (round(ra_c, 4), round(dec_c, 4))

    if key in catalog_cache:
        return catalog_cache[key]

    center = SkyCoord(ra_c * u.deg, dec_c * u.deg)

    simbad = Simbad()
    simbad.TIMEOUT = 60
    simbad.add_votable_fields("ra(d)", "dec(d)")

    try:
        result = simbad.query_region(center, radius=SIMBAD_RADIUS)

        if result is None:
            result = []

        catalog_cache[key] = result

        print(f"[SIMBAD] loaded {len(result)} objects")

        return result

    except Exception as e:
        print("[SIMBAD ERROR]", e)
        catalog_cache[key] = []
        return []


def match_to_catalog(ra, dec, catalog, max_sep_arcsec=5):

    if catalog is None or len(catalog) == 0:
        return ["Unknown"] * len(ra)

    cat_coords = SkyCoord(
        catalog["ra"],
        catalog["dec"],
        unit="deg"
    )

    src_coords = SkyCoord(ra * u.deg, dec * u.deg)

    idx, sep2d, _ = match_coordinates_sky(src_coords, cat_coords)

    names = []

    for i in range(len(ra)):
        if sep2d[i].arcsec < max_sep_arcsec:
            names.append(str(catalog["main_id"][idx[i]]))
        else:
            names.append("Unknown")

    return names


# ----------------------------
# METADATA
# ----------------------------
def get_plate_date(fits_path):
    try:
        header = astrofits.getheader(fits_path)
        for key in ["DATE-OBS", "DATE", "DATEOBS", "MJD-OBS"]:
            if key in header:
                return str(header[key])
        return "Unknown"
    except:
        return "Unknown"


# ----------------------------
# UI
# ----------------------------
progress = widgets.IntProgress(value=0, min=0, max=100, description="Progress:")
status = widgets.Label(value="Ready")
output = widgets.Output()

dropdown = widgets.Dropdown(
    options=[(f.name, f) for f in cutouts],
    description="Plate:"
)


# ----------------------------
# VIEWER
# ----------------------------
def show_plate(change):

    fits_path = change["new"]

    progress.value = 10
    status.value = "Detecting sources..."

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sources, algorithm, data, data_sub, std, x_col, y_col = detect_sources(fits_path)

    progress.value = 40
    status.value = "WCS + catalog..."

    with output:
        clear_output(wait=True)

        if sources is None or len(sources) == 0:
            print("No sources found.")
            return

        hdr = astrofits.getheader(fits_path)
        wcs = WCS(hdr)

        ra, dec = wcs.pixel_to_world_values(
            sources[x_col],
            sources[y_col]
        )

        sources["ra"] = ra
        sources["dec"] = dec

        # ----------------------------
        # SIMBAD catalog (cached)
        # ----------------------------
        catalog = get_plate_catalog(wcs, data.shape)

        object_names = match_to_catalog(
            np.array(ra),
            np.array(dec),
            catalog
        )

        progress.value = 70
        status.value = "Photometry..."

        positions = list(zip(sources[x_col], sources[y_col]))
        apertures = CircularAperture(positions, r=APERTURE_RADIUS)

        phot_table = aperture_photometry(data_sub, apertures)

        flux = np.array(phot_table["aperture_sum"])
        flux = np.where(flux > 0, flux, np.nan)

        inst_mag = -2.5 * np.log10(flux)

        progress.value = 90
        status.value = "Rendering..."

        print(f"Algorithm: {algorithm}")
        print(f"Sources: {len(sources)}")

        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

        axes[0].imshow(data, origin="lower", cmap="gray")
        axes[0].set_title("Raw Plate")

        axes[1].imshow(data, origin="lower", cmap="gray")

        axes[1].scatter(
            sources[x_col],
            sources[y_col],
            s=50,
            facecolors="none",
            edgecolors="red"
        )

        # labels: object + magnitude
        for i, (x, y, mag, name) in enumerate(zip(
            sources[x_col],
            sources[y_col],
            inst_mag,
            object_names
        )):
            if np.isfinite(mag):
                axes[1].text(
                    x + 5,
                    y + 5,
                    f"{name}\n{mag:.2f}",
                    color="yellow",
                    fontsize=6,
                    bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", pad=1)
                )

        plt.tight_layout()
        plt.show()

        progress.value = 100
        status.value = "Done"


# ----------------------------
# INIT
# ----------------------------
dropdown.observe(show_plate, names="value")

display(widgets.VBox([
    dropdown,
    progress,
    status,
    output
]))

show_plate({"new": cutouts[0]})

SIMBAD loaded 56 objects
